# Activity Day 13: Pointers to Objects, Copying, Dynamic Memory

In [ ]:
////character.cpp
// Setup: Character class used throughout this activity
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;

   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_
                  << " lives, " << this->coins_ << " coins\n";
   }
};

int main(){
    Character charstack("Mario")

    this->
    
}

### Question 1

Consider the `Character` class above. Write code that does the following:

**(a)** Create a `Character` object that lives on the stack.

**(b)** Create a pointer to a `Character` object that points to your object on the stack.

**(c)** Create a `Character` object that lives on the heap.

How would the `display()` method be called on each of your three variables? For the pointer variables, show both the dereferencing way and the arrow operator way. Which of your variables do you need to call `delete` on? Why only those?

Using your heap `Character`, give it 100 coins with the arrow operator, display it, and free the memory.

In [ ]:
//// q1.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void collect_coins(int num_coins){ this->coins_ += num_coins; }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
};
int main(){
    // (a) Create a Character on the stack
    Character c("Mario");
    // (b) Create a pointer to your stack Character
    Character* ptrC = &c;
    // (c) Create a Character on the heap
    Character* ptrC2 = new Character("Luigi")
    // Call display() on each variable:
    c.disply();
    ptrC2->display();
    (*ptrC2).display();
    // - stack object:
    
    // - pointer to stack object (both ways):
    
    // - heap object (both ways):
    
    // Give heap Character 100 coins, display, free memory:  
    ptrC2->display();
    // Which do you delete? Do it here:
    delete ptrC2;
    ptrC2 = nullptr;

    
}

### Question 2

Read the following code and **predict the output** before running it. Use a memory diagram to justify your answer.

- `m2` is a **copy** of `m` (separate object).
- `ptr` is a **pointer** to `m` (same object, not a copy).

Predict first, then run to check!

In [1]:
//// q2.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void collect_coins(int num_coins){ this->coins_ += num_coins; }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
};
int main(){
    Character m("Mario");
    Character m2 = m;
    Character* ptr = &m;
    m2.collect_coins(50);
    ptr->collect_coins(100);
    m.display();
    m2.display();
}

wrote file q2.cpp
$> g++   q2.cpp    -o q2
$> ./q2 
Mario: 3 lives, 100 coins
Mario: 3 lives, 50 coins


### Question 3

Suppose we add two new members to the `Character` class:
- `int* history_` -- a heap-allocated array that logs each coin pickup
- `int history_count_` -- how many pickups have been logged

The class also now has `void show_history() const` and a destructor `~Character()` that calls `delete[] this->history_`.

Read the code below. **Before running it**, predict the output. Use a memory diagram to justify.

In [ ]:
//// q3.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   int* history_;
   int history_count_;

   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
        this->history_count_ = 0;
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
       this->history_[this->history_count_] = num_coins;
       this->history_count_++;
   }
   void show_history() const {
       std::cout << this->name_ << "'s coin history: ";
       for(int i = 0; i < this->history_count_; i++){
           std::cout << this->history_[i] << " ";
       }
       std::cout << "\n";
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_
                  << " lives, " << this->coins_ << " coins\n";
   }
   ~Character(){
        std::cout << "Destructor for " << this->name_ << ", history_ at " << this->history_ << "\n";
        delete[] this->history_;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(10);
    m.collect_coins(25);
    Character m2 = m;
    m2.collect_coins(99);

    std::cout << "m.history_ address:  " << m.history_ << "\n";
    std::cout << "m2.history_ address: " << m2.history_ << "\n";

    m.show_history();
    m2.show_history();
}

wrote file q3.cpp
$> g++   q3.cpp    -o q3
$> ./q3 
m.history_ address:  0x5eb88daf12b0
m2.history_ address: 0x5eb88daf12b0
Mario's coin history: 10 25 
Mario's coin history: 10 25 99 
Destructor for Mario, history_ at 0x5eb88daf12b0
Destructor for Mario, history_ at 0x5eb88daf12b0


free(): double free detected in tcache 2
Aborted (core dumped)
executable failed with exit code 134


### Question 3 Follow-Up

Both `m` and `m2` go out of scope at the end of `main()`. Each destructor calls `delete[]` on its `history_` pointer.

**(a)** Which destructor runs first? What code could you add to the destructor method to verify? (Hint: stack objects are destroyed in reverse order of creation.)

**(b)** What goes wrong when the second destructor runs?

### Question 4

The copy constructor signature is given below. Fill in the body so that the new object gets its **OWN** copy of the `history_` array (a deep copy), not just a copy of the pointer.

Then run the test code (same as Q3) to verify that the deep copy works -- the addresses should be different, and modifying one object's history should not affect the other.

In [ ]:
//// q4.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   int* history_;
   int history_count_;

   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
        this->history_ = new int[10];
        for(int i = 0; i < 10; i++) this->history_[i] = 0;
        this->history_count_ = 0;
   }
   // Deep copy constructor -- fill in the body!
   Character(const Character& other){
       // Your code here:
       this->name_ = other.name_;
       this->lives_ = other.lives_;
       this->coins
       
       
       
   }
   void collect_coins(int num_coins){
       this->coins_ += num_coins;
       this->history_[this->history_count_] = num_coins;
       this->history_count_++;
   }
   void show_history() const {
       std::cout << this->name_ << "'s coin history: ";
       for(int i = 0; i < this->history_count_; i++){
           std::cout << this->history_[i] << " ";
       }
       std::cout << "\n";
   }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_
                  << " lives, " << this->coins_ << " coins\n";
   }
   ~Character(){
        delete[] this->history_;
   }
};
int main(){
    Character m("Mario");
    m.collect_coins(10);
    m.collect_coins(25);

    Character m2 = m;
    m2.collect_coins(99);

    std::cout << "m.history_ address:  " << m.history_ << "\n";
    std::cout << "m2.history_ address: " << m2.history_ << "\n";

    m.show_history();
    m2.show_history();
}

### Question 5

*More destructors practice.* Read this code and predict the order in which destructors fire. (This uses a simple `Character` class without `history_` -- just `name_` and a destructor that prints `"Destroying [name]"`.) 

What order are the destructor messages printed? Why?

In [ ]:
//// q5.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   Character(std::string n){ this->name_ = n; }
   ~Character(){ std::cout << "Destroying " << this->name_ << "\n"; }
};
int main(){
    Character a("Alice");
    Character b("Bob");
    Character* c = new Character("Charlie");
    Character d("Diana");
    delete c;
}

### Question 6

**(a)** When you call a function like `void f(int n)` with `f(x)`, something secretly happens before the function body runs. (Hint: there is an *assignment*.) What is it? What does this mean for changes to `n` inside `f`?

In [ ]:
//// q6a.cpp
// What secretly happens when you call f(x)?
#include<iostream>
void f(int n){
    n += 100;
    std::cout << "Inside f: n = " << n << "\n";
}
int main(){
    int x = 5;
    f(x);
    std::cout << "After f: x = " << x << "\n";
}

**(b)** Try the following code and draw a memory diagram. `int& r = x;` makes `r` a **reference** (alias) to `x` -- they are the same variable.

In [ ]:
//// q6b.cpp
#include<iostream>
int main(){
    int x = 5;
    int& r = x;
    r = 99;
    std::cout << "x = " << x << "\n";
    std::cout << "r = " << r << "\n";
    std::cout << "&x = " << &x << ", &r = " << &r << "\n";
}

**(c)** Write a function `addTen(int n)` that adds 10 to its parameter. Call it on `x` in main and print `x` after. Did `x` change? Why not?

In [ ]:
//// q6c.cpp
#include<iostream>
// Write addTen with pass-by-value
void addTen(int n){
    n += 10;
}
int main(){
    int x = 5;
    addTen(x);
    std::cout << "x = " << x << "\n"; // Did x change?
}

**(d)** Rewrite `addTen` using a reference parameter (`int& n`). Does `x` change now? Why?

In [ ]:
//// q6d.cpp
#include<iostream>
// Rewrite addTen using a reference parameter
void addTen(int& n){
    n += 10;
}
int main(){
    int x = 5;
    addTen(x);
    std::cout << "x = " << x << "\n"; // Did x change?
}

**(e)** Using what you've learned, write a `swap` function using references. (Recall: we did this using pointers before.)

In [ ]:
//// q6e.cpp
#include<iostream>
// Write swap using references

int main(){
    int a = 3, b = 7;
    // swap(a, b);
    std::cout << "a = " << a << ", b = " << b << "\n";
}

### Question 7

What happens if you create a `const Character` object and try to call `collect_coins()` on it? What error do you get, and why?

In [ ]:
//// q7.cpp
#include<iostream>
#include<string>
class Character{
   public:
   std::string name_;
   int lives_, coins_;
   Character(std::string n){
        this->name_ = n; this->lives_ = 3; this->coins_ = 0;
   }
   void collect_coins(int num_coins){ this->coins_ += num_coins; }
   void display() const {
        std::cout << this->name_ << ": " << this->lives_ << " lives, " << this->coins_ << " coins\n";
   }
};
int main(){
    const Character m("Mario");
    m.display();       // Does this work?
    // m.collect_coins(10);  // Uncomment this line -- what happens?
}

### Question 8

The copy constructor takes its parameter as `const Character& other`.

**(a)** What would happen if it took the parameter by VALUE instead (i.e., `Character other` instead of `const Character& other`)? Think carefully about what happens when you pass by value...

**(b)** Why is `const` important here? What does it prevent?

Think about this one -- no code needed!